In [1]:
# Install needed libraries
%pip install -U python-jobspy
%pip install tqdm
%pip install xlsxwriter
%pip install tenacity requests

# Install MongoDB Python driver
%pip install pymongo
%pip install python-dotenv

You should consider upgrading via the '/home/wagner/.pyenv/versions/market_scrapper_venv/bin/python -m pip install --upgrade pip' command.


Note: you may need to restart the kernel to use updated packages.


You should consider upgrading via the '/home/wagner/.pyenv/versions/market_scrapper_venv/bin/python -m pip install --upgrade pip' command.


Note: you may need to restart the kernel to use updated packages.


You should consider upgrading via the '/home/wagner/.pyenv/versions/market_scrapper_venv/bin/python -m pip install --upgrade pip' command.


...

Note: you may need to restart the kernel to use updated packages.


You should consider upgrading via the '/home/wagner/.pyenv/versions/market_scrapper_venv/bin/python -m pip install --upgrade pip' command.


Note: you may need to restart the kernel to use updated packages.


You should consider upgrading via the '/home/wagner/.pyenv/versions/market_scrapper_venv/bin/python -m pip install --upgrade pip' command.


Note: you may need to restart the kernel to use updated packages.


In [2]:
import sys
from pathlib import Path

# Get the absolute path of the project root (one level up from the notebooks directory)
project_root = str(Path().resolve().parent)  # Goes up two levels to reach the project root

# Add the project root to the Python path
if project_root not in sys.path:
    sys.path.append(project_root)

In [3]:
from operations import (
    process_and_save_jobs, 
    setup_output_directory, 
    connect_to_mongodb,
    hours_old_since_2025,
    safe_scrape_jobs
)
import itertools

In [4]:
connect_to_mongodb()

Looking for .env at: /home/wagner/Documentos/dev-projects/No Country/Market-Scraper/.env


✅ Successfully connected to MongoDB
📊 Database: job_market
📂 Collection: jobs
🔗 Total documents: 18039


{'client': MongoClient(host=['ac-fky0ob9-shard-00-00.ncfzs7b.mongodb.net:27017', 'ac-fky0ob9-shard-00-02.ncfzs7b.mongodb.net:27017', 'ac-fky0ob9-shard-00-01.ncfzs7b.mongodb.net:27017'], document_class=dict, tz_aware=False, connect=True, appname='Cluster0', authsource='admin', replicaset='atlas-t6534q-shard-0', tls=True, serverselectiontimeoutms=5000),
 'collection': Collection(Database(MongoClient(host=['ac-fky0ob9-shard-00-00.ncfzs7b.mongodb.net:27017', 'ac-fky0ob9-shard-00-02.ncfzs7b.mongodb.net:27017', 'ac-fky0ob9-shard-00-01.ncfzs7b.mongodb.net:27017'], document_class=dict, tz_aware=False, connect=True, appname='Cluster0', authsource='admin', replicaset='atlas-t6534q-shard-0', tls=True, serverselectiontimeoutms=5000), 'job_market'), 'jobs')}

In [5]:
# --- 1. Definir Directorio de Salida ---
output_dir = setup_output_directory("../data/raw")
print(f"Directorio de salida: {output_dir}")

Directorio de salida: ../data/raw/jobs_20251211_100224


## 2. Definir Parámetros de Búsqueda Base

In [6]:
sectores_clave = ["Fintech", "EdTech", "Future of Work"]
search_terms = sectores_clave
hours_old= hours_old_since_2025(2025)
hours_old_list = list(range(0, hours_old, 24))
indeed_glassdoor_countries = [
    "Australia",
    "Austria",
    "Belgium",
    "Brazil",
    "Canada",
    "France",
    "Germany",
    "Hong Kong",
    "India",
    "Ireland",
    "Italy",
    "Mexico",
    "Netherlands",
    "New Zealand",
    "Singapore",
    "Spain",
    "Switzerland",
    "UK",
    "USA",
    "Vietnam"
]

Han pasado 8256 horas desde el 1 de enero de este año.


In [7]:
# --- 3. Lista para guardar resultados ---
# Guardaremos los DataFrames de cada sitio aquí
all_jobs_dfs = []

In [8]:
print("Parámetros listos. Iniciaremos scrapers secuenciales y especializados.")

Parámetros listos. Iniciaremos scrapers secuenciales y especializados.


In [9]:
# --- 1. Scraper: Indeed (El "Caballo de batalla") ---
# Es el más estable y sin límites de solicitudes

print("\n--- Iniciando Scraper: Indeed/Glassdoor ---")
for country_indeed, search_term, hours_old in itertools.product(indeed_glassdoor_countries, search_terms, hours_old_list):
    print(f"Buscando en {country_indeed} por {search_term} de hace {hours_old} horas")
    try:
        indeed_jobs = safe_scrape_jobs(
            site_name=["indeed", "glassdoor"],
            search_term=search_term,
            country_indeed=country_indeed,
            results_wanted=99999,
            hours_old=hours_old
        )
        if indeed_jobs is not None and not indeed_jobs.empty:
            print(f"✅ Se encontraron {len(indeed_jobs)} trabajos.")
            all_jobs_dfs.append(indeed_jobs)
    except Exception as e:
        print(f"❌ Error después de varios intentos: {e}")


--- Iniciando Scraper: Indeed/Glassdoor ---
Buscando en Australia por Fintech de hace 0 horas


2025-12-11 10:03:08,662 - ERROR - JobSpy:Glassdoor - Glassdoor: 'NoneType' object is not iterable


2025-12-11 10:03:12,014 - ERROR - JobSpy:Glassdoor - Glassdoor: bad response status code: 429


✅ Se encontraron 848 trabajos.
Buscando en Australia por Fintech de hace 24 horas


2025-12-11 10:03:13,694 - ERROR - JobSpy:Glassdoor - Glassdoor: bad response status code: 429


...

✅ Se encontraron 114 trabajos.
Buscando en Australia por Fintech de hace 432 horas


2025-12-11 10:04:29,665 - ERROR - JobSpy:Glassdoor - Glassdoor: bad response status code: 429


✅ Se encontraron 117 trabajos.
Buscando en Australia por Fintech de hace 456 horas


✅ Se encontraron 117 trabajos.
Buscando en Australia por Fintech de hace 480 horas


KeyboardInterrupt: 

In [ ]:
"""print("\n--- Iniciando Scraper: Google ---")
for search_term, hours_old in itertools.product(search_terms, hours_old_list):
    print(f"Buscando por {search_term} con {hours_old} horas")
    try:
        google_jobs = safe_scrape_jobs(
            site_name=["google"],
            search_term=search_term,
            google_search_term=f"{search_term}",
            results_wanted=10,
            hours_old=hours_old,
            verbose=2
        )
        if google_jobs is not None and not google_jobs.empty:
            print(f"✅ Se encontraron {len(google_jobs)} trabajos.")
            all_jobs_dfs.append(google_jobs)
        else:
            print(f"❌ No se encontraron trabajos para {search_term}")    
    except Exception as e:
        print(f"❌ Error después de varios intentos: {e}")"""

In [ ]:
"""print("\n--- Iniciando Scraper: ZipRecruiter ---")
for search_term, hours_old in itertools.product(search_terms, hours_old_list):
    print(f"Buscando por {search_term} con {hours_old} horas")
    try:
        zip_jobs = safe_scrape_jobs(
            site_name=["zip_recruiter"],
            search_term=search_term,
            results_wanted=100, 
            hours_old=hours_old,
            verbose=2
        )
        if zip_jobs is not None and not zip_jobs.empty:
            print(f"✅ Se encontraron {len(zip_jobs)} trabajos.")
            all_jobs_dfs.append(zip_jobs)
        else:
            print(f"❌ No se encontraron trabajos para {search_term}")    
    except Exception as e:
        print(f"❌ Error después de varios intentos: {e}")"""

In [ ]:
"""print("\n--- Iniciando Scraper: Bayt, Naukri, BdJobs ---")
for search_term, hours_old in itertools.product(search_terms, hours_old_list):
    print(f"Buscando por {search_term} con {hours_old} horas")
    try:
        bayt_jobs = safe_scrape_jobs(
            site_name=["bayt", "naukri", "bdjobs"],
            search_term=f"{search_term}",
            results_wanted=100, 
            hours_old=hours_old,
            verbose=2
        )
        if bayt_jobs is not None and not bayt_jobs.empty:
            print(f"✅ Se encontraron {len(bayt_jobs)} trabajos.")
            all_jobs_dfs.append(bayt_jobs)
        else:
            print(f"❌ No se encontraron trabajos para {search_term}")    
    except Exception as e:
        print(f"❌ Error después de varios intentos: {e}")"""

In [ ]:
print("\n--- Iniciando Scraper: Linkedin ---")
for search_term, hours_old in itertools.product(search_terms, hours_old_list):
    print(f"Buscando por {search_term} con {hours_old} horas")
    try:
        linkedin_jobs = safe_scrape_jobs(
            site_name=["linkedin"],
            search_term=f"{search_term}",
            results_wanted=99999,
            linkedin_fetch_description=True,
            hours_old=hours_old,
            verbose=2
        )
        if linkedin_jobs is not None and not linkedin_jobs.empty:
            print(f"✅ Se encontraron {len(linkedin_jobs)} trabajos.")
            all_jobs_dfs.append(linkedin_jobs)
        else:
            print(f"❌ No se encontraron trabajos para {search_term}")    
    except Exception as e:
        print(f"❌ Error después de varios intentos: {e}")

In [ ]:
print("\n--- Scraping secuencial completado ---")

In [ ]:
# Update your main processing loop:
if all_jobs_dfs:
    process_and_save_jobs(all_jobs_dfs, output_dir)
else:
    print("\nNo jobs were found.")